# Smart CV Analyzer — Fine-tuning XLM Roberta untuk NER

Pipeline:
1. Setup & instalasi dependensi
2. Parsing dataset format CoNLL
3. Tokenisasi + Subword Alignment
4. Data Collator & Metrik Evaluasi (seqeval)
5. Fine-tuning XLM roberta (`FacebookAI/xlm-roberta-base`)
6. Inference dari file CV asli (PDF / DOCX)

## install & load library

In [36]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(
    "transformers[torch]",
    "datasets",
    "evaluate",
    "seqeval",
    "accelerate>=0.26.0",
    "PyMuPDF",          # fitz — untuk baca PDF
    "python-docx",      # untuk baca DOCX
    "tensorboard"
)

!npm install -g localtunnel

print("done install all dependecies")

⠙⠹⠸⠼⠴
changed 22 packages in 729ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦done install all dependecies


In [38]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import torch
import os
import random
from datasets import Dataset, DatasetDict, ClassLabel, Sequence, Features, Value
from transformers import DataCollatorForTokenClassification
import evaluate
import numpy as np

## load & split dataset CoNLL

In [37]:
INPUT_CONLL_FILE = "/kaggle/input/datasets/laventiliz/cv-text-with-bio-tag-for-ner/dataset_augmented_nodup.conll" 

TRAIN_PATH = "train.conll"
TEST_PATH  = "test.conll"
TEST_SIZE_PERCENT = 0.2

def read_conll_file(filepath):
    """
    Membaca file CoNLL dan mengelompokkannya per kalimat.
    Di format CoNLL, setiap kalimat dipisahkan oleh baris kosong (blank line).
    """
    sentences = []
    current_sentence = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: # Jika baris kosong, berarti satu kalimat selesai
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                current_sentence.append(line)
                
        # Menangkap kalimat terakhir jika file tidak diakhiri baris kosong
        if current_sentence:
            sentences.append(current_sentence)
            
    return sentences

def write_conll_file(sentences, filepath):
    """
    Menulis kembali list kalimat ke dalam format CoNLL.
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for sentence in sentences:
            for line in sentence:
                f.write(line + '\n')
            f.write('\n') # Tambahkan baris kosong antar kalimat

# 1. Pastikan file input ada
if not os.path.exists(INPUT_CONLL_FILE):
    print(f"❌ ERROR: File '{INPUT_CONLL_FILE}' tidak ditemukan di Colab.")
    print("Silakan upload dulu file dataset CoNLL kamu ke menu folder di sebelah kiri.")
else:
    print(f"Membaca dataset dari: {INPUT_CONLL_FILE}...")
    
    # 2. Baca file CoNLL
    semua_kalimat = read_conll_file(INPUT_CONLL_FILE)
    total_kalimat = len(semua_kalimat)
    print(f" Berhasil membaca {total_kalimat} kalimat.")
    
    # 3. Acak (Shuffle) dataset agar distribusinya merata
    random.seed(42) # Set seed agar hasil acakan konsisten/reproducible
    random.shuffle(semua_kalimat)
    
    # 4. Tentukan batas pemotongan (Split)
    split_index = int(total_kalimat * (1 - TEST_SIZE_PERCENT))
    
    train_sentences = semua_kalimat[:split_index]
    test_sentences = semua_kalimat[split_index:]
    
    # 5. Simpan ke file baru
    write_conll_file(train_sentences, TRAIN_PATH)
    write_conll_file(test_sentences, TEST_PATH)
    
    print("\n--- HASIL SPLIT ---")
    print(f" {TRAIN_PATH}: {len(train_sentences)} kalimat ({os.path.getsize(TRAIN_PATH)} bytes)")
    print(f" {TEST_PATH} : {len(test_sentences)} kalimat ({os.path.getsize(TEST_PATH)} bytes)")
    print("done split dataset")

Membaca dataset dari: /kaggle/input/datasets/laventiliz/cv-text-with-bio-tag-for-ner/dataset_augmented_nodup.conll...
 Berhasil membaca 888 kalimat.

--- HASIL SPLIT ---
 train.conll: 710 kalimat (67102 bytes)
 test.conll : 178 kalimat (16627 bytes)
done split dataset


## Parsing Dataset CoNLL to HuggingFace Dataset

In [5]:
from datasets import Dataset, DatasetDict, ClassLabel, Sequence, Features, Value

def parse_conll_file(file_path: str):
    """
    Membaca file .conll dan mengembalikan list of dict:
    [{"tokens": [...], "ner_tags": [...]}, ...]
    
    Format CoNLL yang didukung:
    - Token<TAB>Label   (per baris)
    - Baris kosong      (pemisah kalimat/entri)
    """
    all_tokens, all_labels = [], []
    tokens, labels = [], []

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                # Baris kosong = akhir satu kalimat
                if tokens:
                    all_tokens.append(tokens)
                    all_labels.append(labels)
                    tokens, labels = [], []
            else:
                parts = line.split()   # split by whitespace (tab atau spasi)
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    labels.append(parts[-1])   # ambil kolom terakhir sebagai label
        # Tangani kalimat terakhir jika file tidak diakhiri baris kosong
        if tokens:
            all_tokens.append(tokens)
            all_labels.append(labels)

    return {"tokens": all_tokens, "ner_tags": all_labels}


def build_label_list(train_data, test_data):
    """Kumpulkan semua label unik, pastikan 'O' selalu di index 0."""
    label_set = set()
    for tags in train_data["ner_tags"] + test_data["ner_tags"]:
        label_set.update(tags)
    # 'O' harus index 0 agar alignment mudah
    label_list = ["O"] + sorted(l for l in label_set if l != "O")
    return label_list


def create_hf_dataset(train_path: str, test_path: str):
    """Membuat HuggingFace DatasetDict dari dua file .conll."""
    raw_train = parse_conll_file(train_path)
    raw_test  = parse_conll_file(test_path)

    label_list = build_label_list(raw_train, raw_test)
    label2id   = {l: i for i, l in enumerate(label_list)}

    print(f" Label list ({len(label_list)}): {label_list}")

    # Konversi label string → integer
    train_tags = [[label2id[t] for t in seq] for seq in raw_train["ner_tags"]]
    test_tags  = [[label2id[t] for t in seq] for seq in raw_test["ner_tags"]]

    # Buat HuggingFace Dataset
    features = Features({
        "tokens":   Sequence(Value("string")),
        "ner_tags": Sequence(ClassLabel(names=label_list)),
    })

    train_ds = Dataset.from_dict(
        {"tokens": raw_train["tokens"], "ner_tags": train_tags},
        features=features
    )
    test_ds = Dataset.from_dict(
        {"tokens": raw_test["tokens"], "ner_tags": test_tags},
        features=features
    )

    return DatasetDict({"train": train_ds, "test": test_ds}), label_list


# --- Jalankan ---
raw_datasets, LABEL_LIST = create_hf_dataset(TRAIN_PATH, TEST_PATH)

print("\n Dataset berhasil dibuat:")
print(raw_datasets)
print(f"\nContoh data train[0]:")
print(raw_datasets["train"][0])

 Label list (11): ['O', 'B-CERT', 'B-COMP', 'B-EDU', 'B-ROLE', 'B-SKILL', 'I-CERT', 'I-COMP', 'I-EDU', 'I-ROLE', 'I-SKILL']

 Dataset berhasil dibuat:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 710
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 178
    })
})

Contoh data train[0]:
{'tokens': ['Mahasiswa', 'Universitas', 'Mataram', 'mempelajari', 'Cloud', 'Computing', 'modern'], 'ner_tags': [0, 3, 8, 0, 5, 10, 0]}


## Tokenisasi & Subword Alignment (SANGAT KRUSIAL)

IndoBERT (dan semua model BERT-based) menggunakan **WordPiece tokenizer** yang bisa memecah satu kata menjadi beberapa **subword token**.

**Masalah:**
- Dataset kita berlabel di level **kata** (word-level)
- Model bekerja di level **subword token**

**Contoh:**
Kata asli  : ["Telkom",   "Indonesia"]
Label asli : ["B-ORG",    "I-ORG"   ]
Setelah tokenisasi BERT:
Tokens     : [CLS, "Tel", "##kom", "Indonesia", SEP]

**Solusi — Aturan Alignment:**
1. Token `[CLS]` dan `[SEP]` → label `-100` (diabaikan loss)
2. Token pertama dari sebuah kata → pakai **label asli** kata tersebut
3. Token subword lanjutan (diawali `##`) → label `-100` (diabaikan loss)

Kenapa `-100`? Karena PyTorch `CrossEntropyLoss` secara default mengabaikan index `-100`, sehingga subwords tidak ikut menghitung loss dan tidak mempengaruhi bobot training.

In [6]:
from transformers import AutoTokenizer

#MODEL_CHECKPOINT = "google-bert/bert-base-multilingual-cased"
MODEL_CHECKPOINT = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Tokenizer loaded: {MODEL_CHECKPOINT}")

# Verifikasi subword behaviour
sample_words = ["Universitas", "Gadjah", "Mada"]
encoded = tokenizer(sample_words, is_split_into_words=True)
print(f"\n Contoh subword tokenization:")
print(f"   Kata asli  : {sample_words}")
print(f"   Token IDs  : {encoded['input_ids']}")
print(f"   Tokens     : {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")
print(f"   Word IDs   : {encoded.word_ids()}")


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Token spesial
            else:
                label_ids.append(labels[word_idx])
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


# --- Terapkan ke seluruh dataset (batched untuk efisiensi) ---
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names  # hapus kolom asli
)

print("\n✅ Tokenisasi & alignment selesai.")
print(tokenized_datasets)

# Verifikasi alignment pada satu sampel
sample_idx = 0
sample = raw_datasets["train"][sample_idx]
tok_sample = tokenized_datasets["train"][sample_idx]

print(f"\n Verifikasi alignment sampel {sample_idx}:")
print(f"   Kata asli  : {sample['tokens']}")
print(f"   Label asli : {sample['ner_tags']}")
tokens_decoded = tokenizer.convert_ids_to_tokens(tok_sample["input_ids"])
print(f"   Subwords   : {tokens_decoded}")
print(f"   Labels     : {tok_sample['labels']}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: FacebookAI/xlm-roberta-base

 Contoh subword tokenization:
   Kata asli  : ['Universitas', 'Gadjah', 'Mada']
   Token IDs  : [0, 97374, 99047, 18404, 43706, 2]
   Tokens     : ['<s>', '▁Universitas', '▁Gad', 'jah', '▁Mada', '</s>']
   Word IDs   : [None, 0, 1, 1, 2, None]


Map:   0%|          | 0/710 [00:00<?, ? examples/s]

Map:   0%|          | 0/178 [00:00<?, ? examples/s]


✅ Tokenisasi & alignment selesai.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 710
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 178
    })
})

 Verifikasi alignment sampel 0:
   Kata asli  : ['Mahasiswa', 'Universitas', 'Mataram', 'mempelajari', 'Cloud', 'Computing', 'modern']
   Label asli : [0, 3, 8, 0, 5, 10, 0]
   Subwords   : ['<s>', '▁Mahasiswa', '▁Universitas', '▁Mata', 'ram', '▁mempelajari', '▁Cloud', '▁Comput', 'ing', '▁modern', '</s>']
   Labels     : [-100, 0, 3, 8, 8, 0, 5, 10, 10, 0, -100]


## Data Collator & Metrik Evaluasi

- **`DataCollatorForTokenClassification`**: Menangani **dynamic padding** per batch — lebih efisien daripada padding ke panjang maksimum global. Juga otomatis mengisi label padding dengan `-100`.
- **`seqeval`**: Library evaluasi standar untuk sequence labeling. Menghitung Precision, Recall, F1, dan Accuracy per entitas (bukan per token).

In [7]:
# Data collator: padding dinamis per batch + isi label pad dengan -100
data_collator = DataCollatorForTokenClassification(tokenizer)

# Load metrik seqeval (evaluasi standar NER/sequence labeling)
seqeval_metric = evaluate.load("seqeval")

id2label = {i: l for i, l in enumerate(LABEL_LIST)}
label2id = {l: i for i, l in enumerate(LABEL_LIST)}


def compute_metrics(eval_preds):
    """
    Dipanggil oleh HuggingFace Trainer setelah setiap epoch evaluasi.
    
    Langkah:
    1. Argmax logits → prediksi kelas per token
    2. Abaikan semua posisi dengan label == -100 (subwords & special tokens)
    3. Konversi integer → string label untuk seqeval
    4. Hitung Precision, Recall, F1, Accuracy
    """
    logits, labels = eval_preds

    # Argmax di axis=-1 → shape (batch, seq_len)
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_predictions = [], []

    for pred_seq, label_seq in zip(predictions, labels):
        true_label_seq, true_pred_seq = [], []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                # Abaikan subwords dan special tokens
                continue
            true_label_seq.append(id2label[label_id])
            true_pred_seq.append(id2label[pred_id])

        true_labels.append(true_label_seq)
        true_predictions.append(true_pred_seq)

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }


print("✅ Data collator & fungsi metrik siap.")
print(f"   Jumlah label: {len(LABEL_LIST)}")
print(f"   Label list  : {LABEL_LIST}")

✅ Data collator & fungsi metrik siap.
   Jumlah label: 11
   Label list  : ['O', 'B-CERT', 'B-COMP', 'B-EDU', 'B-ROLE', 'B-SKILL', 'I-CERT', 'I-COMP', 'I-EDU', 'I-ROLE', 'I-SKILL']


## Fine-tuning XLM Roberta

### Konfigurasi Training:
| Parameter | Nilai | Alasan |
|---|---|---|
| Learning Rate | `2e-5` | LR kecil agar tidak merusak pretrained weights |
| Epochs | `5` | Dengan EarlyStopping, akan berhenti jika tidak ada improvement |
| Batch Size | `16` | Sesuai VRAM T4 (16GB); gunakan gradient accumulation jika OOM |
| Warmup Steps | `100` | Hindari LR terlalu tinggi di awal training |
| Weight Decay | `0.01` | Regularisasi ringan |
| EarlyStopping | `patience=3` | Berhenti jika val F1 tidak naik 3 epoch berturut-turut |

In [29]:
import os
import urllib
import time

# 1. Bersihkan zombie process lama (biar fresh)
os.system('pkill -f tensorboard')
os.system('pkill -f localtunnel') 

# 2. Siapkan folder log
log_dir = "/kaggle/working/smart-cv-analyzer-indoroberts/runs" 
os.makedirs(log_dir, exist_ok=True)

# 3. Nyalakan TensorBoard di background
get_ipython().system_raw(f'tensorboard --logdir {log_dir} --host 0.0.0.0 --port 6006 &')
print("✅ TensorBoard menyala di background!")

# 4. Ambil IP untuk password LocalTunnel
ip_password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"🔑 Password/IP kamu: {ip_password}")

# 5. Nyalakan LocalTunnel di background (pakai tanda '&') 
# dan simpan URL-nya ke file teks biar bisa kita baca
get_ipython().system_raw('lt --port 6006 > lt_url.txt 2>&1 &')

print("⏳ Sedang membuat terowongan LocalTunnel...")
time.sleep(5) # Tunggu 5 detik sampai server LocalTunnel membalas

# 6. Baca URL dari file teks yang baru dibuat
try:
    with open('lt_url.txt', 'r') as f:
        url_text = f.read().strip()
        print(f"🌐 Link TensorBoard kamu: {url_text}")
except:
    print("URL belum siap, jalankan cell ini sekali lagi.")

TensorBoard caught SIGTERM; exiting...


✅ TensorBoard menyala di background!
🔑 Password/IP kamu: 35.222.155.234
⏳ Sedang membuat terowongan LocalTunnel...


2026-05-17 03:33:23.488468: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778988803.515207    3244 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778988803.524026    3244 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778988803.544700    3244 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778988803.544773    3244 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778988803.544781    3244 computation_placer.cc:177] computation placer alr

🌐 Link TensorBoard kamu: your url is: https://swift-grapes-kiss.loca.lt



NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

TensorBoard 2.19.0 at http://0.0.0.0:6006/ (Press CTRL+C to quit)


In [30]:
# Deteksi GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu  = torch.cuda.device_count()
print(f"🖥️  Device: {device} | Jumlah GPU: {n_gpu}")

# ── Inisialisasi Model ──────────────────────────────────────
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print(f"\n✅ Model loaded: {MODEL_CHECKPOINT}")
print(f"   Jumlah label output: {model.config.num_labels}")

# ── Training Arguments ──────────────────────────────────────
OUTPUT_DIR = "./smart-cv-analyzer-indoroberts"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # --- Learning schedule ---
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=100,
    weight_decay=0.01,

    # --- Batch size ---
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # --- Evaluasi & Checkpoint ---
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_only_model=True,

    # --- Logging ---
    logging_dir="./runs/ner_experiment_1",
    logging_steps=50,
    report_to="tensorboard",

    # --- Presisi campuran ---
    fp16=torch.cuda.is_available(),
)

# ── Trainer (FIXED) ──────────────────────────────────────────
# transformers >= 4.46: 'tokenizer' diganti 'processing_class'
# transformers <  4.46: ganti kembali ke 'tokenizer=tokenizer'
import transformers
transformers_version = tuple(int(x) for x in transformers.__version__.split(".")[:2])

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Deteksi versi dan pilih argument yang tepat
if transformers_version >= (4, 46):
    trainer_kwargs["processing_class"] = tokenizer
    print(f"ℹ️  transformers {transformers.__version__}: menggunakan 'processing_class'")
else:
    trainer_kwargs["tokenizer"] = tokenizer
    print(f"ℹ️  transformers {transformers.__version__}: menggunakan 'tokenizer'")

trainer = Trainer(**trainer_kwargs)

print("\n🚀 Memulai training...")
train_result = trainer.train()

print("\n✅ Training selesai!")
print(f"   Total steps        : {train_result.global_step}")
print(f"   Training loss akhir: {train_result.training_loss:.4f}")

# ── Simpan model final ──────────────────────────────────────
BEST_MODEL_DIR = "./best-cv-analyzer-model-xlm"
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"\n💾 Model terbaik tersimpan di: {BEST_MODEL_DIR}")

# ── Evaluasi final di test set ──────────────────────────────
print("\n📊 Evaluasi final di test set:")
eval_results = trainer.evaluate()
for key, val in eval_results.items():
    if isinstance(val, float):
        print(f"   {key:30s}: {val:.4f}")

🖥️  Device: cuda | Jumlah GPU: 2


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



✅ Model loaded: FacebookAI/xlm-roberta-base
   Jumlah label output: 11
ℹ️  transformers 5.0.0: menggunakan 'processing_class'

🚀 Memulai training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training selesai!
   Total steps        : 445
   Training loss akhir: 0.9869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Model terbaik tersimpan di: ./best-cv-analyzer-model-xlm

📊 Evaluasi final di test set:


   eval_loss                     : 0.0305
   eval_precision                : 0.9985
   eval_recall                   : 0.9985
   eval_f1                       : 0.9985
   eval_accuracy                 : 0.9977
   eval_runtime                  : 3.2996
   eval_samples_per_second       : 53.9460
   eval_steps_per_second         : 6.9710
   epoch                         : 5.0000


## Inference (PDF / DOCX)

In [15]:
import os
import re

def extract_text_from_cv(file_path: str) -> str:
    ext = os.path.splitext(file_path)[-1].lower()
    raw_text = ""

    if ext == ".pdf":
        import fitz
        doc = fitz.open(file_path)
        n_pages = len(doc)          # ← ambil dulu sebelum close
        pages = []
        for page in doc:
            pages.append(page.get_text("text"))
        doc.close()
        raw_text = "\n".join(pages)
        print(f"📄 PDF berhasil dibaca: {n_pages} halaman")   # ← pakai variabel

    elif ext == ".docx":
        from docx import Document
        doc = Document(file_path)
        paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
        raw_text = "\n".join(paragraphs)
        print(f"📝 DOCX berhasil dibaca: {len(paragraphs)} paragraf")

    else:
        raise ValueError(f"Format file tidak didukung: {ext}. Gunakan .pdf atau .docx")

    # Bersihkan teks
    cleaned = re.sub(r'\n{3,}', '\n\n', raw_text)
    cleaned = "\n".join(line.strip() for line in cleaned.splitlines())
    cleaned = cleaned.strip()

    print(f"✅ Teks berhasil diekstrak: {len(cleaned)} karakter")
    return cleaned


print("✅ Fungsi extract_text_from_cv siap.")

✅ Fungsi extract_text_from_cv siap.


In [33]:
# NER Inference Pipeline
from transformers import pipeline
import torch

# 1. Definisi urutan label yang BENAR (Sesuai dengan output model.config aslimu)
label_list = [
    "O", 
    "B-CERT", "B-COMP", "B-EDU", "B-ROLE", "B-SKILL", 
    "I-CERT", "I-COMP", "I-EDU", "I-ROLE", "I-SKILL"
]

# Buat kamus baru
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# 2. Kembalikan kamus yang benar ke dalam model
model.config.id2label = id2label
model.config.label2id = label2id

model_best = AutoModelForTokenClassification.from_pretrained("/kaggle/working/best-cv-analyzer-model-xlm")
tokenizer_best = AutoTokenizer.from_pretrained("/kaggle/working/best-cv-analyzer-model-xlm", add_prefix_space=True)

# 3. Masukkan kembali model ke dalam pipeline
ner_pipeline = pipeline(
    "token-classification", 
    model=model_best, 
    tokenizer=tokenizer_best, 
    aggregation_strategy="first"
)

print(f"✅ NER Pipeline siap (device: {'GPU' if torch.cuda.is_available() else 'CPU'})")

def preprocess_text(text):
    # Memberikan spasi pada tanda baca agar terpisah dari kata
    text = re.sub(r'([.,()|•])', r' \1 ', text)
    # Menghapus spasi ganda
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_for_dashboard(entities):
    cleaned = []
    # Daftar kata sampah
    garbage = ["TAHUN", "KETIGA", "MAHASISWA"]
    
    for ent in entities:
        # 1. Ambil kata asli
        word = ent['word'].replace(' ', ' ').strip()
        
        # 2. FIX STICKY WORDS: Gunakan Regex untuk memisahkan kata yang nempel
        # Contoh: "FigmaCanva" -> "Figma Canva"
        # Mencari huruf kecil yang diikuti huruf besar: [a-z][A-Z]
        word = re.sub(r'([a-z])([A-Z])', r'\1 \2', word)
        
        # 3. Pisahkan angka yang nempel ke huruf: "September 20232027" -> "September 2023 2027"
        word = re.sub(r'(\d)([A-Z])', r'\1 \2', word)
        word = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', word)

        # 4. Filter kualitas
        if ent['score'] > 0.60 and len(word) > 2:
            # Bersihkan simbol sisa
            word = re.sub(r'[^\w\s&/|]', ' ', word).strip()
            word = re.sub(r'\s+', ' ', word) # Hapus spasi ganda
            
            if word.upper() in garbage:
                continue
                
            cleaned.append({
                "label": ent['entity_group'],
                "text": word,
                "score": ent['score']
            })
    return cleaned

def analyze_cv(text: str):
    # STEP 1: Pre-processing (Regex)
    # Memberikan spasi pada tanda baca agar tidak nempel ke kata
    processed_text = re.sub(r'([.,()|•])', r' \1 ', text)
    processed_text = re.sub(r'\s+', ' ', processed_text).strip()

    # STEP 2: Inference (Chunking)
    words = processed_text.split()
    chunk_size = 400
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    raw_entities = []
    for chunk in chunks:
        if chunk.strip():
            entities = ner_pipeline(chunk)
            raw_entities.extend(entities)

    # STEP 3: Post-processing (Cleaning)
    # Di sinilah kita panggil fungsi pembersih sebelum dikelompokkan
    cleaned_data = clean_for_dashboard(raw_entities)

    # STEP 4: Grouping & Pretty Print
    entity_groups = {}
    for item in cleaned_data:
        label = item["label"]
        if label not in entity_groups:
            entity_groups[label] = []
        
        # Hindari duplikat
        if item["text"] not in [e["word"] for e in entity_groups[label]]:
            entity_groups[label].append({"word": item["text"], "score": item["score"]})

    # Cetak Hasil
    print("\n" + "="*60)
    print("🎯  HASIL ANALISIS CV — Named Entity Recognition")
    print("="*60)

    label_icons = {"EDU": "🎓", "SKILL": "⚙️", "ROLE": "🏷️", "COMP": "🏢", "CERT": "📜"}
    
    for label, items in sorted(entity_groups.items()):
        icon = label_icons.get(label, "🏷️")
        print(f"\n{icon}  {label}")
        print(f"   {'Entitas':<35} {'Confidence':>10}")
        print(f"   {'-'*45}")
        for item in sorted(items, key=lambda x: x["score"], reverse=True):
            print(f"   {item['word']:<35} {item['score']:>9.1%}")

    print("\n" + "="*60)
    return entity_groups


print("✅ Fungsi analyze_cv siap.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ NER Pipeline siap (device: GPU)
✅ Fungsi analyze_cv siap.


In [34]:
# Upload & Analisis CV Asli
CV_FILE_PATH = "/kaggle/input/datasets/laventiliz/cv-text-with-bio-tag-for-ner/CV_31_Wahyu_Dharma.pdf"  # contoh: "/kaggle/input/cv-dataset/cv_saya.pdf"

if CV_FILE_PATH and os.path.exists(CV_FILE_PATH):
    ext = os.path.splitext(CV_FILE_PATH)[-1].lower()

    if ext in [".pdf", ".docx"]:
        # Ekstrak teks dari PDF/DOCX
        cv_text = extract_text_from_cv(CV_FILE_PATH)
        print(f"\n📋 Preview teks (500 karakter pertama):")
        print("-" * 50)
        print(cv_text)
        print("...\n")

        # Analisis NER
        results = analyze_cv(cv_text)

    elif ext == ".txt":
        # Mode fallback untuk file teks biasa
        with open(CV_FILE_PATH, "r", encoding="utf-8") as f:
            cv_text = f.read()
        results = clean_for_dashboard(analyze_cv(cv_text, min_score=0.4))

    else:
        print(f"❌ Format {ext} tidak didukung. Gunakan .pdf, .docx, atau .txt")
else:
    print("ℹ️  Tidak ada file CV yang diset. Menggunakan dummy CV...")
    print("   → Set CV_FILE_PATH ke path file CV kamu, atau uncomment bagian Colab.")
    results = clean_for_dashboard(analyze_cv(cv_text, min_score=0.4))

📄 PDF berhasil dibaca: 1 halaman
✅ Teks berhasil diekstrak: 1422 karakter

📋 Preview teks (500 karakter pertama):
--------------------------------------------------
Wahyu Dharma
Mobile Developer (Android/iOS)
Surabaya | +62 886-1046-5542 | wahyu.dharma19@yahoo.com | linkedin.com/in/wahyu-dharma | github.com/wahyudha
RINGKASAN PROFIL
Mobile developer yang berpengalaman dalam membangun aplikasi native dan cross-platform untuk Android dan
iOS. Memiliki rekam jejak dalam mengembangkan aplikasi dengan pengguna aktif jutaan. Fokus pada performa
aplikasi dan pengalaman pengguna yang optimal.
KEAHLIAN TEKNIS
React Native • SQLite • Flutter • Swift • Jetpack Compose • Kotlin • CI/CD • Git • REST API • SwiftUI
PENGALAMAN KERJA
Mobile Developer (Android/iOS) — Kudo (Grab)
2022 – 2024
• Memimpin migrasi sistem monolitik ke arsitektur microservices untuk 3 layanan utama.
• Mengurangi bug production sebesar 50% melalui implementasi CI/CD yang ketat.
• Mengintegrasikan sistem pembayaran dengan 4 paym

In [18]:
# Jalankan pipeline mentah 
hasil_mentah = analyze_cv("Calon Data Scientist dan AI Engineer dengan latar belakang pendidikan S1 Teknik Informatika yang mendalami spesialisasi Machine Learning dan AI. Memiliki ketertarikan kuat dalam memanfaatkan teknologi baru untuk menyelesaikan tantangan keberlanjutan. Berpengalaman dalam siklus hidup proyek data secara penuh, mulai dari pra-pemrosesan hingga visualisasi, serta antusias untuk menerapkan keterampilan analitis dan pemecahan masalah yang kuat guna mengembangkan solusi teknologi inovatif bagi peningkatan bisnis. Saat ini sedang meningkatkan keterampilan yang relevan dengan industri melalui bootcamp yang terafiliasi dengan Accenture dan berupaya memberikan kontribusi pada misi perusahaan dalam mengintegrasikan keberlanjutan pada solusi klien") 


🎯  HASIL ANALISIS CV — Named Entity Recognition

🏢  COMP
   Entitas                             Confidence
   ---------------------------------------------
   Accenture                               96.4%

🎓  EDU
   Entitas                             Confidence
   ---------------------------------------------
   S 1 Teknik Informatika                  99.9%

🏷️  ROLE
   Entitas                             Confidence
   ---------------------------------------------
   Data Scientist                          99.9%
   AIEngineer                              99.5%

⚙️  SKILL
   Entitas                             Confidence
   ---------------------------------------------
   Machine Learning                        99.7%
   visualisasi                             99.6%
   analitis                                99.6%
   keberlanjutan                           99.5%
   pra pemrosesan                          97.9%
   masalah                                 88.3%
   teknologibaru           

In [41]:
# Teks dari bagian Experience dan Projects CV Fahalliza
text_exp_proj = """
AI Division Member | Informatics Study Club, UIN Sunan Kalijaga. 
Optimize LLM performance using Retrieval-Augmented Generation (RAG) techniques.
Design and manage 20+ visual content pieces for social media.
RFM-Based Customer Segmentation. Developed an interactive Streamlit dashboard 
for customer segmentation using RFM analysis and clustering methods (K-Means, GMM).
TikTok Review Sentiment Analysis. Developed an end-to-end NLP pipeline to classify 
Google Play Store reviews using modified JSON dictionary to normalize Indonesian slang words.
"""

# Jalankan analisis
print("🚀 Menganalisis Pengalaman & Proyek...")
results_complex = analyze_cv(text_exp_proj)

🚀 Menganalisis Pengalaman & Proyek...

🎯  HASIL ANALISIS CV — Named Entity Recognition

📜  CERT
   Entitas                             Confidence
   ---------------------------------------------
   Google                                  90.5%
   Store                                   62.4%

🎓  EDU
   Entitas                             Confidence
   ---------------------------------------------
   UINSunan Kalijaga                       99.7%

🏷️  ROLE
   Entitas                             Confidence
   ---------------------------------------------
   AIDivision Member                       99.9%
   end to end NLP                          83.8%
   customer                                66.3%

⚙️  SKILL
   Entitas                             Confidence
   ---------------------------------------------
   K Means                                 99.9%
   GMM                                     99.9%
   Retrieval Augmented Generation          99.7%
   clustering                         

In [32]:
import os
from collections import Counter, defaultdict

CONLL_FILE = "/kaggle/input/datasets/laventiliz/cv-text-with-bio-tag-for-ner/dataset_augmented_nodup.conll"

# =========================================
# LOAD DATASET CONLL
# =========================================

sentences = []
current_sentence = []

with open(CONLL_FILE, "r", encoding="utf-8") as f:
    for line in f:

        line = line.strip()

        if line == "":
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
            continue

        parts = line.split("\t")

        if len(parts) != 2:
            continue

        token, tag = parts
        current_sentence.append((token, tag))

if current_sentence:
    sentences.append(current_sentence)

print(f"✅ Total kalimat: {len(sentences)}")


# =========================================
# VALIDATOR BIO
# =========================================

print("\n" + "="*60)
print("🔍 VALIDASI BIO")
print("="*60)

bio_errors = []

for sent_id, sentence in enumerate(sentences):

    prev_tag = "O"

    for idx, (token, tag) in enumerate(sentence):

        if tag.startswith("I-"):

            entity_type = tag[2:]

            # I-tag tidak boleh muncul tanpa B-tag
            if prev_tag == "O":
                bio_errors.append(
                    f"Kalimat {sent_id+1}: I-{entity_type} muncul tanpa B- ({token})"
                )

            elif prev_tag[2:] != entity_type:
                bio_errors.append(
                    f"Kalimat {sent_id+1}: Entity mismatch ({token})"
                )

        prev_tag = tag

if bio_errors:
    print(f"❌ Total BIO Error: {len(bio_errors)}")

    for err in bio_errors[:20]:
        print("-", err)

else:
    print("✅ Tidak ada BIO error")


# =========================================
# DISTRIBUSI LABEL
# =========================================

print("\n" + "="*60)
print("📊 DISTRIBUSI LABEL")
print("="*60)

label_counter = Counter()

for sentence in sentences:
    for token, tag in sentence:
        label_counter[tag] += 1

for label, count in sorted(label_counter.items()):
    print(f"{label:15} : {count}")


# =========================================
# ENTITY DISTRIBUTION
# =========================================

print("\n" + "="*60)
print("🏷️ DISTRIBUSI ENTITY")
print("="*60)

entity_counter = Counter()

for label, count in label_counter.items():

    if label.startswith("B-"):
        entity_counter[label[2:]] += count

for ent, count in entity_counter.items():
    print(f"{ent:15} : {count}")


# =========================================
# IMBALANCE CHECK
# =========================================

print("\n" + "="*60)
print("⚖️ CHECK IMBALANCE")
print("="*60)

if entity_counter:

    max_entity = max(entity_counter.values())
    min_entity = min(entity_counter.values())

    imbalance_ratio = max_entity / max(min_entity, 1)

    print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

    if imbalance_ratio > 5:
        print("⚠️ Dataset sangat tidak seimbang")
    else:
        print("✅ Distribusi entity cukup seimbang")


# =========================================
# DETECT WEIRD BOUNDARY
# =========================================

print("\n" + "="*60)
print("🧠 DETECT ANNOTATION BOUNDARY ANEH")
print("="*60)

weird_entities = []

for sentence in sentences:

    entity_tokens = []
    entity_type = None

    for token, tag in sentence:

        if tag.startswith("B-"):

            if entity_tokens:
                entity_text = " ".join(entity_tokens)

                # suspicious
                if len(entity_text) <= 3:
                    weird_entities.append(entity_text)

                if entity_text.endswith("-"):
                    weird_entities.append(entity_text)

            entity_tokens = [token]
            entity_type = tag[2:]

        elif tag.startswith("I-"):
            entity_tokens.append(token)

        else:

            if entity_tokens:
                entity_text = " ".join(entity_tokens)

                if len(entity_text) <= 3:
                    weird_entities.append(entity_text)

                if entity_text.endswith("-"):
                    weird_entities.append(entity_text)

            entity_tokens = []

print(f"⚠️ Total suspicious entities: {len(weird_entities)}")

for ent in weird_entities[:30]:
    print("-", ent)


# =========================================
# TOKEN STATISTICS
# =========================================

print("\n" + "="*60)
print("📈 TOKEN STATISTICS")
print("="*60)

total_tokens = 0

for sentence in sentences:
    total_tokens += len(sentence)

avg_len = total_tokens / len(sentences)

print(f"Total token     : {total_tokens}")
print(f"Rata-rata token : {avg_len:.2f}")


# =========================================
# ENTITY EXAMPLES
# =========================================

print("\n" + "="*60)
print("🧪 SAMPLE ENTITY")
print("="*60)

entity_examples = defaultdict(set)

for sentence in sentences:

    current_entity = []
    current_type = None

    for token, tag in sentence:

        if tag.startswith("B-"):

            if current_entity:
                entity_examples[current_type].add(
                    " ".join(current_entity)
                )

            current_entity = [token]
            current_type = tag[2:]

        elif tag.startswith("I-"):
            current_entity.append(token)

        else:

            if current_entity:
                entity_examples[current_type].add(
                    " ".join(current_entity)
                )

            current_entity = []
            current_type = None

for ent_type, examples in entity_examples.items():

    print(f"\n{ent_type}")

    for ex in list(examples)[:10]:
        print(" -", ex)


print("\n✅ VALIDASI SELESAI")

✅ Total kalimat: 888

🔍 VALIDASI BIO
✅ Tidak ada BIO error

📊 DISTRIBUSI LABEL
B-CERT          : 255
B-COMP          : 200
B-EDU           : 276
B-ROLE          : 572
B-SKILL         : 691
I-CERT          : 518
I-COMP          : 28
I-EDU           : 462
I-ROLE          : 892
I-SKILL         : 491
O               : 1587

🏷️ DISTRIBUSI ENTITY
ROLE            : 572
COMP            : 200
SKILL           : 691
EDU             : 276
CERT            : 255

⚖️ CHECK IMBALANCE
Imbalance Ratio: 3.46
✅ Distribusi entity cukup seimbang

🧠 DETECT ANNOTATION BOUNDARY ANEH
⚠️ Total suspicious entities: 8
- OVO
- OVO
- BCA
- BCA
- OVO
- OVO
- OVO
- OVO

📈 TOKEN STATISTICS
Total token     : 5972
Rata-rata token : 6.73

🧪 SAMPLE ENTITY

ROLE
 - Scrum Master
 - Backend API Consultant
 - AI Solutions Architect
 - Frontend Programmer
 - Cyber Risk Specialist
 - Cloud Solutions Architect
 - Frontend Interactive Designer
 - Cloud System Administrator
 - Cloud Engineering Manager
 - DevOps Automation Speciali